In [1]:
# STANDIN A0
# Load the data, the split and the shared helpers (parts/00_setup.ipynb). Dropped from the hand-in
# notebook, where the setup cells come first.
%run 00_setup.ipynb

Python 3.13.7, scikit-learn 1.6.1, numpy 2.5.3, pandas 3.0.6
Working directory: /home/steven/Desktop/AI-for-Cybersecurity-Final-Mini-Project
Data folder: /home/steven/Desktop/AI-for-Cybersecurity-Final-Mini-Project/UNSW-NB15
score() and score_proba() ready


Rows with an infinite or missing value: 0
Loaded 447,915 flows x 76 features


Raw rows:                      447,915
Exact duplicates removed:      141,742
Conflicting rows removed:        1,068
Clean rows (all are used):     305,105


Dropped 9 constant columns: Bwd PSH Flags, Fwd URG Flags, Bwd URG Flags, URG Flag Count, CWR Flag Count, ECE Flag Count, Fwd Bytes/Bulk Avg, Fwd Packet/Bulk Avg, Fwd Bulk Rate Avg
67 features: 65 continuous, 2 0/1 columns (Fwd PSH Flags, RST Flag Count)
TUNE_IDX: 36,612 training rows for settings searches
Setup complete: 22 shared names ready in 65 s


In [2]:
# STANDIN B2
# The Part B black box with the settings chosen on validation in B2 (parts/10_baseline_models.ipynb).
# Dropped from the hand-in notebook, where the real B2 cell comes first.
from sklearn.ensemble import HistGradientBoostingClassifier

BOOST_SETTINGS = {"learning_rate": 0.1, "max_leaf_nodes": 31, "random_state": SEED}
def make_boost():
    return HistGradientBoostingClassifier(**BOOST_SETTINGS)

boost = make_boost().fit(X_train, y_train)

In [3]:
# STANDIN D1
# The global SHAP ranking of step D1 (parts/30_explanation.ipynb): 1,000 random validation flows, seed 42.
# Dropped from the hand-in notebook, where the real D1 cell comes first.
import shap

GLOSSARY = pd.read_csv(TABLES / "T4_feature_glossary.csv", index_col="feature")
explainer = shap.TreeExplainer(boost)
sv = explainer(X_val.sample(1000, random_state=SEED))
shap_global = (pd.Series(np.abs(sv.values).mean(axis=0), index=FEATURES)
               .sort_values(ascending=False))

<!-- STEP F1 -->
## Part F: a knowledge-based layer (belief rule base)

A **belief rule base (BRB)** decides with a handful of readable rules instead of hundreds of trees:

> IF `feature 1` is *High* AND `feature 2` is *Low* THEN the risk is Low 10%, Medium 30%, High 60%.

Each input feature gets three **referential values**, Low, Medium and High. A flow's value is matched
against them (a value halfway between Low and Medium is half Low, half Medium), the rules fire according
to how well they match, and **Evidential Reasoning** combines their beliefs into one answer, including an
explicit **Unknown** part when the rules cannot decide. The brief asks for **two features with three
referential values each, so 3 × 3 = 9 rules**. The engine is Lab 3's tested `src/brbes.py`.

**Step F1: which two features?** The brief says: the two most important features from Part D. We go
down the global SHAP ranking (D1, computed on validation flows, so this choice never looks at the test
set) and keep the first two features that can carry three **distinct** referential values. These are
the training data's 5th percentile (Low), median (Medium) and 95th percentile (High). A feature is
skipped if it is a 0/1 flag, or if two of those values coincide and `brbes.make_levels` cannot separate
them (when many flows share one value, it first tries placing Medium and High among the values above
Low).

In [4]:
# STEP F1
if "src" not in sys.path:
    sys.path.insert(0, "src")
import brbes  # noqa: E402  the BRB engine from Lab 3 (src/brbes.py)

rows, TOP2 = [], []
for rank, feature in enumerate(shap_global.index, start=1):
    row = {"rank": rank, "feature": feature, "shap_share": shap_global[feature] / shap_global.sum(),
           "distinct_values": int(X_train[feature].nunique()),
           "percentiles_5_50_95": X_train[feature].quantile([0.05, 0.50, 0.95]).round(2).tolist()}
    if feature in FLAG_COLS:
        row.update(status="skipped", reason="0/1 flag: cannot have three levels")
    else:
        try:
            levels = brbes.make_levels(X_train[feature])
            row.update(status="chosen", reason="three distinct levels", levels=np.round(levels, 2).tolist())
            TOP2.append(feature)
        except ValueError:
            row.update(status="skipped", reason="two of the three levels coincide, even among the values above Low")
    rows.append(row)
    if len(TOP2) == 2:
        break

f1 = pd.DataFrame(rows).set_index("rank")
f1.to_csv(TABLES / "F1_feature_choice.csv")
print(f"TOP2 = {TOP2}")
f1

TOP2 = ['FWD Init Win Bytes', 'Bwd Packets/s']


,feature,shap_share,distinct_values,percentiles_5_50_95,status,reason,levels
rank,,,,,,,
1,Fwd Seg Size Min,0.420203,7,"[8.0, 32.0, 32.0]",skipped,"two of the three levels coincide, even among t...",NaN
2,FWD Init Win Bytes,0.214088,534,"[0.0, 5792.0, 26064.0]",chosen,three distinct levels,"[0.0, 5792.0, 26064.0]"
3,Bwd Packets/s,0.129336,132000,"[5.13, 296.67, 5633.8]",chosen,three distinct levels,"[5.13, 296.67, 5633.8]"


<!-- STEP F1 -->
### The two features for the BRB

| SHAP rank | Feature | Share of SHAP weight | Low / Medium / High (training 5th percentile / median / 95th percentile) |
|---:|---|---:|---|
| 1 | `Fwd Seg Size Min` | 42% | skipped: 8 / 32 / 32, only 7 distinct values, Medium and High coincide |
| 2 | **`FWD Init Win Bytes`** | 21% | **0 / 5,792 / 26,064** bytes |
| 3 | **`Bwd Packets/s`** | 13% | **5.1 / 296.7 / 5,633.8** packets per second |

`TOP2 = ["FWD Init Win Bytes", "Bwd Packets/s"]`.

**Why the most important feature is skipped.** `Fwd Seg Size Min` takes only 7 values, and 32 is both its
median and its 95th percentile (most benign flows have 32). Three distinct reference points cannot be
placed on it, not even among the values above its lowest value, so the next two features in the ranking
are used.

**What the two features mean.**
- **`FWD Init Win Bytes`**: the TCP window the initiating machine announces when it opens the connection,
  a setting of that machine's operating system (the second half of the attacker fingerprint, D1). In the
  E4 groups it is **Free**: the attacker can change it.
- **`Bwd Packets/s`**: how fast the other side sends packets back. It describes the victim's reaction
  (scans and floods get few or no replies) and is **Fixed**: the attacker cannot set it directly.

So the BRB will reason with one fingerprint feature and one behaviour feature.

**A limitation to watch in F2 and F3.** The attacker machines' window, 16,383 bytes, lies between Medium
(5,792) and High (26,064). The BRB reads it as about 48% Medium and 52% High, the same as any other value
near 16,000. Unlike the trees, which test for a narrow interval around 16,383 (B1), the BRB cannot single
out this exact value with these three reference points.